In [ ]:
# -*- coding: utf-8 -*-
"""
resample_v2.py
==============
Resamples LIDC-IDRI CT images and lung masks to isotropic 1×1×1 mm spacing.

Source repos  (read-only):
  Images : hourouu/LIDC_IDRI_PROCESSED   →  images/*.nii.gz
  Masks  : hourouu/LIDC_IDRI_MASKS       →  masks/*.nii.gz

Destination repo:
  hourouu/LIDC_IDRI_res
    ├── images/*.nii.gz   (resampled CT,   sitkLinear,          fill = -1000)
    └── masks/*.nii.gz    (resampled mask, sitkNearestNeighbor, fill =     0)

Key corrections vs v1
---------------------
1. Image and mask are resampled in the same function, sharing identical
   output geometry (size, spacing, origin, direction) so they are always
   perfectly aligned.
2. DefaultPixelValue = -1000 for images (air), 0 for masks (background).
3. Mask uses sitkNearestNeighbor — no fractional blending on binary labels.
4. Stems are matched between the two source repos so a missing mask causes
   a clean skip rather than a silent mismatch.
"""

# ── 0. Dependencies ────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "SimpleITK", "huggingface_hub", "tqdm"])

# ── 1. Imports ─────────────────────────────────────────────────────────────────
import gc
import io
import json
import re
import shutil
import time
import traceback
import threading
from pathlib import Path
from queue import Queue, Empty

import SimpleITK as sitk
from huggingface_hub import (
    CommitOperationAdd, HfApi, HfFileSystem,
    hf_hub_download,
)
from huggingface_hub.utils import EntryNotFoundError

# ── 2. Configuration ───────────────────────────────────────────────────────────
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Source repos
IMG_SRC_REPO  = "hourouu/LIDC_IDRI_PROCESSED"
MSK_SRC_REPO  = "hourouu/LIDC_IDRI_MASKS"
IMG_SRC_DIR   = "images"
MSK_SRC_DIR   = "masks"

# Destination repo
DST_REPO      = "hourouu/LIDC_IDRI_res"
IMG_DST_DIR   = "images"
MSK_DST_DIR   = "masks"

TARGET_SPACING   = (1.0, 1.0, 1.0)   # mm  (x, y, z)
UPLOAD_BATCH_SIZE = 8                 # pairs per commit  (= 16 files)
PROGRESS_FILE     = "progress.json"   # lives on DST_REPO

WORK_DIR    = Path("/tmp/lidc_resample")
STAGING_DIR = Path("/tmp/lidc_staging")
for d in (WORK_DIR, STAGING_DIR / IMG_DST_DIR, STAGING_DIR / MSK_DST_DIR):
    d.mkdir(parents=True, exist_ok=True)

api = HfApi(token=HF_TOKEN)
fs  = HfFileSystem(token=HF_TOKEN)

# ── 3. Progress ledger ─────────────────────────────────────────────────────────

def _ledger_path_on_hub() -> str:
    return PROGRESS_FILE

def load_progress() -> set:
    """Return set of already-processed stems from the Hub ledger."""
    try:
        local = hf_hub_download(
            repo_id=DST_REPO, repo_type="dataset",
            filename=_ledger_path_on_hub(), token=HF_TOKEN,
            force_download=True,
        )
        with open(local) as f:
            data = json.load(f)
        done = set(data.get("done", []))
        print(f"  Ledger pulled — {len(done)} pair(s) already processed.")
        return done
    except EntryNotFoundError:
        print("  No ledger yet — starting fresh.")
        return set()
    except Exception as exc:
        print(f"  WARNING: could not pull ledger ({exc}) — starting fresh.")
        return set()

def save_progress_bytes(done: set) -> bytes:
    return json.dumps({"done": sorted(done)}, indent=2).encode()

# ── 4. Hub discovery ───────────────────────────────────────────────────────────

def list_stems(repo_id: str, folder: str, suffix: str) -> dict:
    """
    Return {stem: repo_relative_path} for all matching files in repo/folder.
    stem = filename with suffix stripped  (e.g. "LIDC-IDRI-0001")
    """
    pattern = f"datasets/{repo_id}/{folder}/*{suffix}"
    files   = fs.glob(pattern)
    prefix  = f"datasets/{repo_id}/"
    result  = {}
    for f in files:
        rel  = f[len(prefix):]
        stem = Path(rel).name.replace(suffix, "")
        result[stem] = rel
    print(f"  {repo_id}/{folder}: {len(result)} file(s) found.")
    return result

def get_done_stems_on_dst() -> set:
    """Stems that already have BOTH image and mask in the destination repo."""
    try:
        imgs = set(list_stems(DST_REPO, IMG_DST_DIR, "_img.nii.gz").keys())
        msks = set(list_stems(DST_REPO, MSK_DST_DIR, "_msk.nii.gz").keys())
        return imgs & msks
    except Exception:
        return set()

# ── 5. Core resampling — image + mask together ─────────────────────────────────

def resample_pair(
    img_sitk: sitk.Image,
    msk_sitk: sitk.Image,
    target_spacing: tuple = TARGET_SPACING,
) -> tuple[sitk.Image, sitk.Image]:
    """
    Resample a CT image and its binary mask to target_spacing.

    Both outputs share IDENTICAL geometry (size, spacing, origin, direction)
    so they are guaranteed to be perfectly aligned after resampling.

    Image  → sitkLinear        + fill -1000  (air, correct CT background)
    Mask   → sitkNearestNeighbor + fill 0    (background = not-lung)
    """
    orig_spacing = img_sitk.GetSpacing()   # (sx, sy, sz)
    orig_size    = img_sitk.GetSize()      # (nx, ny, nz)

    # New size that preserves physical extent at the target spacing
    new_size = [
        int(round(sz * sp / ts))
        for sz, sp, ts in zip(orig_size, orig_spacing, target_spacing)
    ]

    # --- Shared geometry parameters (derived from the IMAGE) ---
    direction = img_sitk.GetDirection()
    origin    = img_sitk.GetOrigin()

    # --- Image resampler ---
    f_img = sitk.ResampleImageFilter()
    f_img.SetOutputSpacing(target_spacing)
    f_img.SetSize(new_size)
    f_img.SetOutputDirection(direction)
    f_img.SetOutputOrigin(origin)
    f_img.SetTransform(sitk.Transform())
    f_img.SetDefaultPixelValue(-1000)            # air — correct CT border fill
    f_img.SetInterpolator(sitk.sitkLinear)       # smooth interpolation for HU
    resampled_img = f_img.Execute(img_sitk)

    # --- Mask resampler — identical geometry, different interpolator ---
    f_msk = sitk.ResampleImageFilter()
    f_msk.SetOutputSpacing(target_spacing)
    f_msk.SetSize(new_size)                      # same as image
    f_msk.SetOutputDirection(direction)          # from IMAGE, not mask
    f_msk.SetOutputOrigin(origin)                # from IMAGE, not mask
    f_msk.SetTransform(sitk.Transform())
    f_msk.SetDefaultPixelValue(0)                # background = not-lung
    f_msk.SetInterpolator(sitk.sitkNearestNeighbor)  # binary — no blending
    resampled_msk = f_msk.Execute(msk_sitk)

    return resampled_img, resampled_msk


def process_stem(stem: str, img_repo_path: str, msk_repo_path: str) -> tuple[Path, Path]:
    """
    Download one image+mask pair, resample them together, save locally.
    Returns (local_img_path, local_msk_path).
    """
    img_fname = Path(img_repo_path).name
    msk_fname = Path(msk_repo_path).name

    # Local output paths
    out_img = STAGING_DIR / IMG_DST_DIR / img_fname
    out_msk = STAGING_DIR / MSK_DST_DIR / msk_fname

    # Download image
    dl_img = hf_hub_download(
        repo_id=IMG_SRC_REPO, repo_type="dataset",
        filename=img_repo_path, token=HF_TOKEN,
        local_dir=str(WORK_DIR), local_dir_use_symlinks=False,
    )
    # hf_hub_download may nest the file; find it
    dl_img = Path(dl_img)
    if not dl_img.exists():
        matches = list(WORK_DIR.rglob(img_fname))
        if not matches:
            raise FileNotFoundError(f"Downloaded image not found: {img_fname}")
        dl_img = matches[0]

    # Download mask
    dl_msk = hf_hub_download(
        repo_id=MSK_SRC_REPO, repo_type="dataset",
        filename=msk_repo_path, token=HF_TOKEN,
        local_dir=str(WORK_DIR), local_dir_use_symlinks=False,
    )
    dl_msk = Path(dl_msk)
    if not dl_msk.exists():
        matches = list(WORK_DIR.rglob(msk_fname))
        if not matches:
            raise FileNotFoundError(f"Downloaded mask not found: {msk_fname}")
        dl_msk = matches[0]

    # Load
    img_sitk = sitk.ReadImage(str(dl_img))
    msk_sitk = sitk.ReadImage(str(dl_msk))

    orig_spacing = img_sitk.GetSpacing()
    orig_size    = img_sitk.GetSize()

    # Resample both together
    resampled_img, resampled_msk = resample_pair(img_sitk, msk_sitk)

    new_size = resampled_img.GetSize()
    print(
        f"  [R] {stem}  "
        f"{orig_size}@{[round(s,2) for s in orig_spacing]}mm"
        f"  →  {new_size}@{list(TARGET_SPACING)}mm"
    )

    # Save
    sitk.WriteImage(resampled_img, str(out_img), useCompression=True)
    sitk.WriteImage(resampled_msk, str(out_msk), useCompression=True)

    # Cleanup downloads immediately to save disk
    for p in (dl_img, dl_msk):
        try:
            p.unlink()
        except Exception:
            pass

    del img_sitk, msk_sitk, resampled_img, resampled_msk
    gc.collect()

    return out_img, out_msk

# ── 6. Batch upload ────────────────────────────────────────────────────────────

def commit_batch(
    local_imgs: list[Path], local_msks: list[Path],
    done: set, batch_idx: int,
) -> bool:
    """
    Upload all images and masks for a batch + updated ledger in one commit.
    Returns True on success.
    """
    operations = []
    for lp in local_imgs:
        operations.append(CommitOperationAdd(
            path_in_repo=f"{IMG_DST_DIR}/{lp.name}",
            path_or_fileobj=str(lp),
        ))
    for lp in local_msks:
        operations.append(CommitOperationAdd(
            path_in_repo=f"{MSK_DST_DIR}/{lp.name}",
            path_or_fileobj=str(lp),
        ))
    # Ledger inline
    operations.append(CommitOperationAdd(
        path_in_repo=PROGRESS_FILE,
        path_or_fileobj=io.BytesIO(save_progress_bytes(done)),
    ))

    for attempt in range(3):
        try:
            api.create_commit(
                repo_id=DST_REPO, repo_type="dataset",
                commit_message=f"Resampled batch {batch_idx + 1} — {len(local_imgs)} pair(s)",
                operations=operations,
                token=HF_TOKEN,
            )
            print(f"  [U] ✓ Batch {batch_idx + 1} committed ({len(local_imgs)} pair(s)).")
            return True
        except Exception as e:
            err = str(e)
            if "429" in err or "rate limit" in err.lower():
                wait = 70
                m = re.search(r"Retry after (\d+) seconds", err)
                if m:
                    wait = int(m.group(1)) + 5
                print(f"  [U] Rate limited — waiting {wait}s…")
                time.sleep(wait)
            else:
                print(f"  [U] ✗ Commit failed: {e}")
                traceback.print_exc()
                return False

    print("  [U] ✗ All retry attempts exhausted.")
    return False

# ── 7. Main ────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print(f"Images source : {IMG_SRC_REPO}/{IMG_SRC_DIR}/")
    print(f"Masks  source : {MSK_SRC_REPO}/{MSK_SRC_DIR}/")
    print(f"Destination   : {DST_REPO}")
    print(f"Target spacing: {TARGET_SPACING} mm")
    print("=" * 60)

    # Pull ledger + reconcile with what's actually on the dst repo
    done = load_progress()
    done |= get_done_stems_on_dst()

    # Discover source files
    print("\nDiscovering source files…")
    img_stems = list_stems(IMG_SRC_REPO, IMG_SRC_DIR, "_img.nii.gz")
    msk_stems = list_stems(MSK_SRC_REPO, MSK_SRC_DIR, "_msk.nii.gz")

    # Only process stems that have BOTH an image and a mask
    paired = sorted(set(img_stems) & set(msk_stems))
    missing_mask  = set(img_stems) - set(msk_stems)
    missing_image = set(msk_stems) - set(img_stems)

    if missing_mask:
        print(f"  WARNING: {len(missing_mask)} image(s) have no matching mask — will skip.")
    if missing_image:
        print(f"  WARNING: {len(missing_image)} mask(s) have no matching image — will skip.")

    todo = [s for s in paired if s not in done]
    print(f"\n  {len(todo)} pair(s) to process  ({len(done)} already done).")

    if not todo:
        print("\n✅ Nothing left to process.")
        return

    batches = [todo[i:i + UPLOAD_BATCH_SIZE] for i in range(0, len(todo), UPLOAD_BATCH_SIZE)]
    print(f"  → {len(batches)} batch(es) of up to {UPLOAD_BATCH_SIZE} pair(s) each.\n")

    for batch_idx, batch in enumerate(batches):
        print(f"{'─' * 60}")
        print(f"BATCH {batch_idx + 1}/{len(batches)}  ({len(batch)} pairs)")
        print(f"{'─' * 60}")

        local_imgs, local_msks, batch_done = [], [], []

        for stem in batch:
            try:
                out_img, out_msk = process_stem(
                    stem,
                    img_stems[stem],
                    msk_stems[stem],
                )
                local_imgs.append(out_img)
                local_msks.append(out_msk)
                batch_done.append(stem)
            except Exception as exc:
                print(f"  ✗ FAILED {stem}: {exc}")
                traceback.print_exc()

        if not local_imgs:
            print(f"  No pairs produced in batch {batch_idx + 1} — skipping commit.")
            continue

        # Update done set before committing so ledger is correct on success
        done |= set(batch_done)

        ok = commit_batch(local_imgs, local_msks, done, batch_idx)

        if ok:
            for p in local_imgs + local_msks:
                try:
                    p.unlink()
                except Exception:
                    pass
        else:
            # Roll back so failed pairs are retried next run
            done -= set(batch_done)
            print(f"  ⚠ Commit failed — local files kept for retry.")

    print("\n" + "=" * 60)
    print("Done.")
    print(f"  Processed this run : {len(done)}")
    remaining = len([s for s in paired if s not in done])
    print(f"  Remaining          : {remaining}")
    print("=" * 60)


if __name__ == "__main__":
    main()

Images source : hourouu/LIDC_IDRI_PROCESSED/images/
Masks  source : hourouu/LIDC_IDRI_MASKS/masks/
Destination   : hourouu/LIDC_IDRI_res
Target spacing: (1.0, 1.0, 1.0) mm


progress.json: 0.00B [00:00, ?B/s]

  Ledger pulled — 988 pair(s) already processed.

Discovering source files…
  hourouu/LIDC_IDRI_PROCESSED/images: 996 file(s) found.
  hourouu/LIDC_IDRI_MASKS/masks: 996 file(s) found.

  8 pair(s) to process  (988 already done).
  → 1 batch(es) of up to 8 pair(s) each.

────────────────────────────────────────────────────────────
BATCH 1/1  (8 pairs)
────────────────────────────────────────────────────────────


images/LIDC-IDRI-0226_img.nii.gz:   0%|          | 0.00/35.4M [00:00<?, ?B/s]

masks/LIDC-IDRI-0226_msk.nii.gz:   0%|          | 0.00/502k [00:00<?, ?B/s]

...

  ...LIDC-IDRI-0231_msk.nii.gz: 100%|##########|  403kB /  403kB            

  ...LIDC-IDRI-0232_msk.nii.gz: 100%|##########|  251kB /  251kB            

  ...LIDC-IDRI-0227_img.nii.gz:  12%|#2        | 7.97MB / 63.8MB            

  ...LIDC-IDRI-0226_img.nii.gz:  11%|#1        | 7.99MB / 70.2MB            

  [U] ✓ Batch 1 committed (8 pair(s)).

Done.
  Processed this run : 996
  Remaining          : 0


In [ ]:
import os, subprocess

hf_cache = os.path.expanduser("~/.cache/huggingface/hub")

if not os.path.exists(hf_cache):
    print("HuggingFace cache is empty.")
else:
    print("HuggingFace cached datasets/models:\n")
    entries = sorted(os.listdir(hf_cache))
    total = 0
    for entry in entries:
        full_path = os.path.join(hf_cache, entry)
        result = subprocess.run(["du", "-sb", full_path], capture_output=True, text=True)
        size = int(result.stdout.split()[0]) if result.stdout else 0
        total += size
        print(f"  {size/1e9:.2f} GB   {entry}")
    print(f"\n  TOTAL : {total/1e9:.2f} GB")

HuggingFace cache is empty.
